# Notebook 06 — Longitudinal Progress Tracker
Re-run as new data arrives. Tracks improvement over time.

| Audience | What they get |
|---|---|
| **User / Patient** | Your progress trend — are you improving? |
| **Doctor / Clinician** | Slope of recovery, plateau detection, dropout flags |
| **App maker** | Session frequency and whether users return — retention signal |


In [1]:
DATA_DIR    = '.'
OUT_DIR     = 'outputs'
USER_LABEL  = 'Patient A'
ROLLING     = 3
import os, json, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

# ── Make sure output folder exists BEFORE anything tries to write to it ──
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, os.path.dirname(os.path.abspath('hci_utils.py')))
from hci_utils import (load_board_tries, load_bd_sessions, load_piano_sessions,
                        load_piano_movements, compare_groups, save_fig,
                        SHAPE_ORDER, HAND_COLORS, GROUP_COLORS)

df_tries         = load_board_tries(DATA_DIR)
bd_valid, bd_bad = load_bd_sessions(DATA_DIR)
piano_sess, _    = load_piano_sessions(DATA_DIR)
piano_valid      = piano_sess[piano_sess['is_valid']].copy().reset_index(drop=True)

print(f'Board drawing sessions (valid): {len(bd_valid)}  Piano sessions (valid): {len(piano_valid)}')


KeyError: 'startedAt'

## Fig 06a — Board drawing score over sessions

In [ ]:
bd_valid = bd_valid.reset_index(drop=True)
bd_valid['session_num']   = range(1, len(bd_valid) + 1)
bd_valid['rolling_score'] = bd_valid['sessionScore'].rolling(ROLLING, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(bd_valid['session_num'], bd_valid['sessionScore'],
        'o--', color='#2c4fa0', alpha=0.5, label='Raw score')
ax.plot(bd_valid['session_num'], bd_valid['rolling_score'],
        lw=2.5, color='#2c4fa0', label='Rolling avg (n=' + str(ROLLING) + ')')

if len(bd_valid) >= 2:
    from scipy import stats as sp
    sl, ic, rv, pv, _ = sp.linregress(bd_valid['session_num'], bd_valid['sessionScore'])
    xl = np.array([1, len(bd_valid)])
    ax.plot(xl, ic + sl * xl, '--', color='#c84b2f', lw=1.5,
            label='Trend (r²=' + f'{rv**2:.2f}' + ', p=' + f'{pv:.2f})')

ax.set_xlabel('Session number')
ax.set_ylabel('Session score')
ax.set_title(USER_LABEL + ' — Board drawing score over time', fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig(fig, 'fig06a_bd_score_over_sessions.png', OUT_DIR)
plt.show()

print('\n=== USER FEEDBACK ===')
if len(bd_valid) >= 2:
    direction = 'improving' if sl > 0 else 'declining'
    print(f'Your board drawing score is {direction} at {abs(sl):.1f} pts/session')
print('\n=== CLINICIAN SIGNAL ===')
print('Flat trend after initial improvement = plateau. Consider increasing difficulty.')
print('Sudden score drop = possible fatigue or symptom flare. Flag for review.')


## Fig 06b — Piano score over sessions

In [ ]:
piano_valid['session_num'] = range(1, len(piano_valid) + 1)
piano_valid['rolling']     = piano_valid['sessionScore'].rolling(ROLLING, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(piano_valid['session_num'], piano_valid['sessionScore'],
        'o--', color='#ba7517', alpha=0.5, label='Raw score')
ax.plot(piano_valid['session_num'], piano_valid['rolling'],
        lw=2.5, color='#ba7517', label='Rolling avg (n=' + str(ROLLING) + ')')

if len(piano_valid) >= 2:
    from scipy import stats as sp
    sl, ic, rv, pv, _ = sp.linregress(piano_valid['session_num'], piano_valid['sessionScore'])
    xl = np.array([1, len(piano_valid)])
    ax.plot(xl, ic + sl * xl, '--', color='#c84b2f', lw=1.5,
            label='Trend (r²=' + f'{rv**2:.2f}' + ')')

ax.set_xlabel('Session number')
ax.set_ylabel('Session score')
ax.set_title(USER_LABEL + ' — Piano score over time', fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig(fig, 'fig06b_piano_score_over_sessions.png', OUT_DIR)
plt.show()


## Fig 06c — Piano response time over sessions

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(piano_valid['session_num'], piano_valid['avg_rt'],
        'o-', color='#2a6e4f', lw=2)
ax.set_xlabel('Session number')
ax.set_ylabel('Avg response time (s)')
ax.set_title(
    USER_LABEL + ' — Piano response time over time'
    + '\n(decreasing = faster reflexes = improvement)',
    fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig06c_response_time_over_sessions.png', OUT_DIR)
plt.show()

print('\n=== CLINICIAN SIGNAL ===')
if len(piano_valid) >= 2:
    rt_start = piano_valid['avg_rt'].iloc[0]
    rt_end   = piano_valid['avg_rt'].iloc[-1]
    delta    = rt_end - rt_start
    print(f'RT change: {rt_start:.2f}s -> {rt_end:.2f}s  (delta={delta:+.2f}s)')
    print(f'{"Improvement" if delta < 0 else "Slower"} in response speed')
print('\n=== USER FEEDBACK ===')
print('A falling line here = your reaction speed is getting faster. Great progress!')


## Fig 06d — Hand asymmetry gap over sessions

In [ ]:
df_sorted = df_tries.sort_values('startedAt').reset_index(drop=True)
window_size = 10
asym_rows = []
for start in range(0, len(df_sorted), window_size):
    chunk = df_sorted.iloc[start:start + window_size]
    left_chunk  = chunk[chunk['hand'] == 'Left']
    right_chunk = chunk[chunk['hand'] == 'Right']
    if len(left_chunk) == 0 or len(right_chunk) == 0:
        continue
    l = left_chunk['completed'].mean() * 100
    r = right_chunk['completed'].mean() * 100
    asym_rows.append({'window': start // window_size + 1, 'asym': l - r})

asym_df = pd.DataFrame(asym_rows).dropna()

if asym_df.empty:
    print('Not enough tries for asymmetry window analysis yet.')
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(asym_df['window'], asym_df['asym'], 'o-', color='#2c4fa0', lw=2)
    ax.axhline(0, color='black', lw=1, ls='--', label='No asymmetry')
    ax.fill_between(asym_df['window'], asym_df['asym'], 0,
                    where=(asym_df['asym'] > 0).values,
                    alpha=0.15, color='#2a6e4f', label='Left better')
    ax.fill_between(asym_df['window'], asym_df['asym'], 0,
                    where=(asym_df['asym'] < 0).values,
                    alpha=0.15, color='#c84b2f', label='Right better')
    ax.set_xlabel('Try window (each window = ' + str(window_size) + ' tries)')
    ax.set_ylabel('Left - Right success rate (pp)')
    ax.set_title(
        USER_LABEL + ' — Hand asymmetry over time'
        + '\n(gap moving toward 0 = impaired hand recovering)',
        fontweight='bold')
    ax.legend()
    plt.tight_layout()
    save_fig(fig, 'fig06d_hand_gap_over_sessions.png', OUT_DIR)
    plt.show()


## Fig 06e — Session performance heatmap

In [ ]:
records = []
for _, row in bd_valid.iterrows():
    records.append({'session': 'BD-' + str(_+1), 'type': 'Board Drawing',
                    'score': row['sessionScore'], 'date': row['time']})
for i, row in piano_valid.iterrows():
    records.append({'session': 'Piano-' + str(i+1), 'type': 'Piano',
                    'score': row['sessionScore'], 'date': row['time']})

if records:
    rec_df = pd.DataFrame(records).sort_values('date').reset_index(drop=True)
    for t in rec_df['type'].unique():
        mask = rec_df['type'] == t
        mn = rec_df.loc[mask, 'score'].min()
        mx = rec_df.loc[mask, 'score'].max()
        rec_df.loc[mask, 'norm_score'] = (rec_df.loc[mask, 'score'] - mn) / (mx - mn + 1e-9)

    pivot = rec_df.pivot_table(index='session', columns='type', values='norm_score')
    fig, ax = plt.subplots(figsize=(6, max(3, len(pivot) * 0.5 + 1)))
    im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    plt.colorbar(im, ax=ax, label='Normalised score per game type')
    ax.set_title(USER_LABEL + ' — Session performance heatmap', fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'fig06e_session_heatmap.png', OUT_DIR)
    plt.show()
    print('\n=== CLINICIAN SIGNAL ===')
    print('Green rows across all columns = good overall session.')
    print('Red rows = poor session — correlate with date for context (illness, fatigue).')


## Export longitudinal summary CSV

In [ ]:
summary_rows = []
for _, row in bd_valid.iterrows():
    summary_rows.append({'session_date': str(row.get('time',''))[:10],
                         'game': 'board_drawing', 'score': row['sessionScore']})
for _, row in piano_valid.iterrows():
    summary_rows.append({'session_date': str(row.get('time',''))[:10],
                         'game': 'piano', 'score': row['sessionScore'],
                         'hit_rate': row['hit_rate'], 'avg_rt': row['avg_rt']})

pd.DataFrame(summary_rows).to_csv(
    os.path.join(OUT_DIR, 'longitudinal_summary.csv'), index=False)
print('longitudinal_summary.csv saved')
